# 04 · Attention mechanisms

Dissects SE, channel attention, spatial attention and CBAM, then looks at the
ablation the training pipeline produced.


In [1]:
# Make the project's own modules importable, and apply the OpenMP fix that must
# run before torch is imported.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "training"))

import app.core.runtime  # noqa: F401  (sets OMP env before numpy/torch)

import json
import numpy as np
import matplotlib.pyplot as plt

from app.core.config import settings

print("Project root:", PROJECT_ROOT)


Project root: D:\Data science projects\Plant disease detection


## The four blocks

| Block | Gate shape | Question it answers |
|---|---|---|
| SE | `(B, C, 1, 1)` | *What* matters — from average pooling alone |
| CBAM channel | `(B, C, 1, 1)` | *What* matters — from average **and** max pooling |
| CBAM spatial | `(B, 1, H, W)` | *Where* it matters |
| CBAM | both, in sequence | Both, channel first |


In [2]:
import torch
from app.ml.attention import CBAM, ChannelAttention, SEBlock, SpatialAttention, build_attention

x = torch.randn(2, 64, 28, 28)
print(f"{'block':<10}{'output':<22}{'params':>8}")
print("-" * 40)
for name in ("none", "se", "channel", "spatial", "cbam"):
    block = build_attention(name, 64)
    out = block(x)
    params = sum(p.numel() for p in block.parameters())
    print(f"{name:<10}{str(tuple(out.shape)):<22}{params:>8,}")


block     output                  params
----------------------------------------
none      (2, 64, 28, 28)              0
se        (2, 64, 28, 28)            512
channel   (2, 64, 28, 28)            512
spatial   (2, 64, 28, 28)             98
cbam      (2, 64, 28, 28)            610


## CBAM is exactly `x * channel_gate * spatial_gate`

Verified numerically rather than asserted.

In [3]:
cbam = CBAM(64)
out = cbam(x)
expected = x * cbam.last_channel_attention * cbam.last_spatial_attention
print("channel gate:", tuple(cbam.last_channel_attention.shape))
print("spatial gate:", tuple(cbam.last_spatial_attention.shape))
print("out == x * channel * spatial :", torch.allclose(out, expected, atol=1e-5))


channel gate: (2, 64, 1, 1)
spatial gate: (2, 1, 28, 28)
out == x * channel * spatial : True


## The learned spatial gate on a real leaf

Read straight out of the forward pass of the trained model.

In [4]:
from PIL import Image
from app.ml.datasets import read_manifest
from app.ml.explain import cbam_spatial_map, colorize, overlay_heatmap
from app.ml.transforms import build_eval_transform, resized_rgb
from evaluate import load_checkpoint

manifest_path = settings.exported_dir / "production.json"
cbam_checkpoint = None
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    candidates = [manifest] + manifest.get("candidates", [])
    cbam_checkpoint = next((c for c in candidates if "cbam" in c["model_name"]), None)

if cbam_checkpoint:
    path = PROJECT_ROOT / cbam_checkpoint["checkpoint"]
    model, config, class_names = load_checkpoint(path, torch.device("cpu"))
    rows = read_manifest(settings.data_dir / "manifests" / "test_bench.csv")

    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for column, row in enumerate(rows[::700][:4]):
        with Image.open(row.path) as handle:
            image = handle.convert("RGB")
        tensor = build_eval_transform(config.image_size)(image).unsqueeze(0)
        result = cbam_spatial_map(model, tensor)
        base = np.asarray(resized_rgb(image, config.image_size))

        axes[0, column].imshow(base)
        axes[0, column].set_title(row.class_name.replace("___", "\n").replace("_", " "), fontsize=7)
        axes[1, column].imshow(overlay_heatmap(base, result.heatmap))
        axes[1, column].set_title("CBAM spatial gate", fontsize=8)
        for ax in (axes[0, column], axes[1, column]):
            ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No CBAM model exported yet. Run the training suite and export_model.py.")


C:\Users\ayush\AppData\Local\Temp\ipykernel_17128\477731332.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Ablation results

Five arms, one backbone, one budget, one seed. Only the attention block differs.


In [5]:
path = settings.results_dir / "ablation_summary.json"
if path.exists():
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(f"{'ARM':<5}{'CONFIGURATION':<34}{'ACC':>9}{'MACRO F1':>11}{'Δ F1':>10}{'PARAMS':>11}")
    print("-" * 80)
    for arm in summary["arms"]:
        delta = arm["delta_vs_control"]["f1_macro"]
        delta_text = "—" if arm["arm"] == "A" else f"{delta * 100:+.2f}pt"
        print(f"{arm['arm']:<5}{arm['label'][:33]:<34}{arm['accuracy']:>9.4f}"
              f"{arm['f1_macro']:>11.4f}{delta_text:>10}{arm['parameters']:>11,}")

    verdict = summary.get("verdict")
    if verdict:
        print("\nVERDICT")
        print(" ", verdict["statement"])
        print("\nCAVEAT")
        print(" ", verdict["caveat"])
else:
    print("Run `python training/run_experiments.py --suite ablation` then `analyse_results.py`.")


ARM  CONFIGURATION                           ACC   MACRO F1      Δ F1     PARAMS
--------------------------------------------------------------------------------
A    CNN (control)                        0.9588     0.9582         —  1,248,774
B    CNN + Channel attention              0.9634     0.9631   +0.49pt  1,259,782
C    CNN + Spatial attention              0.9612     0.9606   +0.24pt  1,249,166
D    CNN + SE                             0.9638     0.9634   +0.51pt  1,259,782
E    CNN + CBAM                           0.9629     0.9628   +0.45pt  1,260,174

VERDICT
  CBAM improved macro F1 by 0.45 points, which is within the 2.14-point run-to-run variation measured on this setup. The improvement is directionally positive but not established.

CAVEAT
  All five arms share one architecture, one training budget, one data subset and one seed; only the attention block differs. A single run per arm cannot establish statistical significance. Retraining cnn_baseline at 3 seeds moved macro 